# Bank Loan Prediction with a Decision Tree

**Course activity:** Cross-validation, loss minimization, OLS and gradient descent

| Member | Section |
|---|---|
| Member 1: _name_ | Cross-validation |
| Member 2: _name_ | Loss minimization |
| Member 3: _name_ | Ordinary Least Squares |
| Member 4: _name_ | Gradient descent and final model |

**Dataset:** `bankloan.csv`, 5,000 bank customers. The target is `Personal.Loan` (1 = accepted the loan offer). Only 9.6% of customers said yes, so the classes are imbalanced.

**Question:** Can a decision tree tell us which customers will accept a personal loan, and how do we choose its depth without fooling ourselves?

For the loss, OLS and gradient descent sections we use a simple regression inside the same data: predicting `CCAvg` (monthly credit card spend, $000) from `Income` ($000).

## Setup

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import plotly.graph_objects as go

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

COLORS = ["#2563eb", "#f97316", "#10b981", "#e11d48", "#8b5cf6"]


class LoanTreeLab:
    def __init__(self, path, target="Personal.Loan", seed=42):
        df = pd.read_csv(path)
        df["Experience"] = df["Experience"].clip(lower=0)
        self.df = df.drop(columns=["ID", "ZIP.Code"])
        self.target = target
        self.seed = seed
        X = self.df.drop(columns=target)
        y = self.df[target]
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=seed
        )
        self.education = {"All": None, "Undergrad": 1, "Graduate": 2, "Advanced": 3}

    def cross_validation(self, depths=range(1, 16), k=5):
        folds = StratifiedKFold(n_splits=k, shuffle=True, random_state=self.seed)
        metrics = ["f1", "accuracy", "roc_auc"]
        rows = []
        for criterion in ["gini", "entropy"]:
            for d in depths:
                tree = DecisionTreeClassifier(max_depth=d, criterion=criterion, random_state=self.seed)
                s = cross_validate(tree, self.X_train, self.y_train, cv=folds,
                                   scoring=metrics, return_train_score=True)
                for m in metrics:
                    rows.append({"criterion": criterion, "max_depth": d, "metric": m,
                                 "train": s[f"train_{m}"].mean(), "cv": s[f"test_{m}"].mean(),
                                 "cv_std": s[f"test_{m}"].std()})
        results = pd.DataFrame(rows)

        grid = GridSearchCV(DecisionTreeClassifier(random_state=self.seed),
                            {"max_depth": list(depths), "criterion": ["gini", "entropy"]},
                            cv=folds, scoring="f1").fit(self.X_train, self.y_train)
        self.best_tree = grid.best_estimator_

        combos = [(c, m) for c in ["gini", "entropy"] for m in metrics]
        fig = go.Figure()
        for c, m in combos:
            sub = results[(results.criterion == c) & (results.metric == m)]
            show = (c, m) == combos[0]
            fig.add_trace(go.Scatter(x=sub.max_depth, y=sub.train, name="Train", mode="lines+markers",
                                     line=dict(width=3, color=COLORS[0]), visible=show))
            fig.add_trace(go.Scatter(x=sub.max_depth, y=sub.cv, name=f"{k}-fold CV", mode="lines+markers",
                                     line=dict(width=3, color=COLORS[1]), visible=show,
                                     error_y=dict(type="data", array=sub.cv_std, thickness=1.5)))
        buttons = []
        for i, (c, m) in enumerate(combos):
            vis = [j // 2 == i for j in range(len(combos) * 2)]
            buttons.append(dict(label=f"{c} | {m}", method="update",
                                args=[{"visible": vis}, {"yaxis.title.text": m}]))
        fig.add_vline(x=grid.best_params_["max_depth"], line_dash="dash", line_color="gray",
                      annotation_text=f"GridSearchCV pick: depth {grid.best_params_['max_depth']}")
        fig.update_layout(template="plotly_white", height=480, title="Train vs cross-validated score by tree depth",
                          xaxis_title="max_depth", yaxis_title="f1", hovermode="x unified",
                          updatemenus=[dict(buttons=buttons, x=0, y=1.18, xanchor="left")])
        fig.show()

        table = (results.pivot_table(index="max_depth", columns=["criterion", "metric"], values="cv")
                 .round(4))
        return table, grid

    def loss_minimization(self, feature="Income", target="CCAvg", n=60, outlier=40.0):
        w0 = np.linspace(-10, 10, 801)
        fig = make_subplots(rows=1, cols=2, specs=[[{"secondary_y": True}, {}]],
                            subplot_titles=(f"Constant model on {target}: loss vs w₀",
                                                            f"Decision stump: SSE vs {feature} threshold"))
        groups = list(self.education.items())
        summary = []
        per_group = 6
        for g, (label, edu) in enumerate(groups):
            data = self.df if edu is None else self.df[self.df["Education"] == edu]
            x, y = data[feature].to_numpy(float), data[target].to_numpy(float)
            rng = np.random.default_rng(self.seed)
            ys = rng.choice(y, size=min(n, len(y)), replace=False)
            yo = np.append(ys, outlier)
            show = g == 0
            curves = {
                "MSE": ((ys[:, None] - w0) ** 2).mean(axis=0),
                "MAE": np.abs(ys[:, None] - w0).mean(axis=0),
                "MSE + outlier": ((yo[:, None] - w0) ** 2).mean(axis=0),
                "MAE + outlier": np.abs(yo[:, None] - w0).mean(axis=0),
            }
            for i, (name, loss) in enumerate(curves.items()):
                fig.add_trace(go.Scatter(x=w0, y=loss, name=name, visible=show, legendgroup=name,
                                         line=dict(width=3, color=COLORS[i % 2],
                                                   dash="dot" if "outlier" in name else "solid")),
                              row=1, col=1, secondary_y=name.startswith("MAE"))

            xs = np.unique(x)
            thresholds = (xs[:-1] + xs[1:]) / 2
            sse = np.array([((y[x <= t] - y[x <= t].mean()) ** 2).sum() +
                            ((y[x > t] - y[x > t].mean()) ** 2).sum() for t in thresholds])
            best = thresholds[sse.argmin()]
            sk = DecisionTreeRegressor(max_depth=1).fit(x.reshape(-1, 1), y).tree_.threshold[0]
            fig.add_trace(go.Scatter(x=thresholds, y=sse, name="SSE", visible=show,
                                     line=dict(width=3, color=COLORS[2])), row=1, col=2)
            fig.add_trace(go.Scatter(x=[best], y=[sse.min()], name="Best split", mode="markers",
                                     marker=dict(size=14, color=COLORS[3], symbol="star"), visible=show),
                          row=1, col=2)

            summary.append({"education": label,
                            "MSE best w0": w0[curves["MSE"].argmin()],
                            "MSE best w0 (outlier)": w0[curves["MSE + outlier"].argmin()],
                            "MAE best w0": w0[curves["MAE"].argmin()],
                            "MAE best w0 (outlier)": w0[curves["MAE + outlier"].argmin()],
                            "stump threshold": best, "sklearn threshold": sk})

        fig.update_yaxes(title_text="MSE", row=1, col=1, secondary_y=False)
        fig.update_yaxes(title_text="MAE", row=1, col=1, secondary_y=True, showgrid=False)
        fig.update_yaxes(title_text="SSE", row=1, col=2)
        buttons = [dict(label=label, method="update",
                        args=[{"visible": [j // per_group == g for j in range(len(groups) * per_group)]}])
                   for g, (label, _) in enumerate(groups)]
        fig.update_layout(template="plotly_white", height=500, hovermode="x unified",
                          xaxis_title="w₀", xaxis2_title=feature,
                          updatemenus=[dict(buttons=buttons, x=0, y=1.2, xanchor="left")],
                          title=dict(text="Education filter", x=0.12, y=0.97, font=dict(size=13)))
        fig.show()
        return pd.DataFrame(summary).round(3)

    def normal_equation(self, feature="Income", target="CCAvg", max_degree=8):
        x, y = self.df[feature].to_numpy(float), self.df[target].to_numpy(float)
        z = (x - x.mean()) / x.std()

        X = np.column_stack([np.ones_like(z), z, z ** 2])
        beta = np.linalg.inv(X.T @ X) @ X.T @ y
        sk = LinearRegression().fit(X[:, 1:], y)
        coefs = pd.DataFrame({"normal equation": beta, "sklearn": np.r_[sk.intercept_, sk.coef_]},
                             index=["intercept", "z", "z²"]).round(6)

        rows = []
        grid = np.linspace(z.min(), z.max(), 300)
        fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Polynomial OLS fit of {target}",
                                                            "Condition number of XᵀX"))
        fig.add_trace(go.Scattergl(x=z, y=y, mode="markers", name="data",
                                   marker=dict(size=4, color="lightgray")), row=1, col=1)
        for d in range(1, max_degree + 1):
            Xr = np.column_stack([x ** p for p in range(d + 1)])
            Xs = np.column_stack([z ** p for p in range(d + 1)])
            b_inv = np.linalg.inv(Xr.T @ Xr) @ Xr.T @ y
            b_lst = np.linalg.lstsq(Xr, y, rcond=None)[0]
            b_s = np.linalg.lstsq(Xs, y, rcond=None)[0]
            rows.append({"degree": d,
                         "cond raw": np.linalg.cond(Xr.T @ Xr), "cond scaled": np.linalg.cond(Xs.T @ Xs),
                         "MSE inv() raw": np.mean((Xr @ b_inv - y) ** 2),
                         "MSE lstsq raw": np.mean((Xr @ b_lst - y) ** 2)})
            fig.add_trace(go.Scatter(x=grid, y=np.column_stack([grid ** p for p in range(d + 1)]) @ b_s,
                                     name=f"degree {d}", visible=d == 2,
                                     line=dict(width=4, color=COLORS[0])), row=1, col=1)
        cond = pd.DataFrame(rows)
        fig.add_trace(go.Scatter(x=cond.degree, y=cond["cond raw"], name="raw x", mode="lines+markers",
                                 line=dict(width=3, color=COLORS[3])), row=1, col=2)
        fig.add_trace(go.Scatter(x=cond.degree, y=cond["cond scaled"], name="standardized x",
                                 mode="lines+markers", line=dict(width=3, color=COLORS[2])), row=1, col=2)

        steps = []
        for d in range(1, max_degree + 1):
            vis = [True] + [k == d for k in range(1, max_degree + 1)] + [True, True]
            steps.append(dict(label=str(d), method="update", args=[{"visible": vis}]))
        fig.update_layout(template="plotly_white", height=500,
                          sliders=[dict(active=1, steps=steps, currentvalue=dict(prefix="Degree: "), len=0.45)],
                          xaxis_title=f"{feature} (standardized)", yaxis_title=target,
                          xaxis2_title="degree", yaxis2=dict(type="log", title="cond(XᵀX)"))
        fig.show()

        tree = DecisionTreeRegressor(max_depth=2).fit(x.reshape(-1, 1), y)
        leaves = tree.apply(x.reshape(-1, 1))
        leaf_check = []
        for leaf in np.unique(leaves):
            ones = np.ones((np.sum(leaves == leaf), 1))
            ols = (np.linalg.inv(ones.T @ ones) @ ones.T @ y[leaves == leaf])[0]
            leaf_check.append({"leaf": leaf, "points": len(ones),
                               "tree value": tree.tree_.value[leaf][0][0], "intercept-only OLS": ols})
        return coefs, cond, pd.DataFrame(leaf_check).round(6)

    def gradient_descent(self, feature="Income", target="CCAvg", lrs=(0.001, 0.01, 0.1, 0.5),
                         epochs=60, batch=8):
        x, y = self.df[feature].to_numpy(float), self.df[target].to_numpy(float)
        z = (x - x.mean()) / x.std()
        X = np.column_stack([np.ones_like(z), z])
        n = len(y)
        ols = np.linalg.lstsq(X, y, rcond=None)[0]

        w = np.zeros(2)
        for it in range(1, 20001):
            grad = 2 / n * X.T @ (X @ w - y)
            if np.linalg.norm(grad) < 1e-6:
                break
            w -= 0.1 * grad
        check = pd.DataFrame({"batch GD": w, "OLS": ols}, index=["intercept", "slope"]).round(4)

        variants = {"Batch": n, f"Mini-batch ({batch})": batch, "SGD": 1, "SGD + schedule": 1}
        fig = make_subplots(rows=1, cols=2, subplot_titles=("Training loss (log scale)", "Batch GD path on the loss surface"))
        a, b = np.meshgrid(np.linspace(ols[0] - 3, ols[0] + 3, 80), np.linspace(ols[1] - 3, ols[1] + 3, 80))
        surface = ((y[:, None] - (a.ravel() + b.ravel() * z[:, None])) ** 2).mean(axis=0).reshape(a.shape)
        fig.add_trace(go.Contour(x=a[0], y=b[:, 0], z=surface, colorscale="Blues", showscale=False,
                                 contours=dict(coloring="lines"), line=dict(width=1.5), name="loss"), row=1, col=2)
        board = []
        per_lr = len(variants) + 1
        for li, lr in enumerate(lrs):
            for vi, (name, size) in enumerate(variants.items()):
                rng = np.random.default_rng(self.seed)
                w = np.zeros(2)
                path, losses, t = [w.copy()], [], 0
                for _ in range(epochs):
                    order = rng.permutation(n)
                    for start in range(0, n, size):
                        idx = order[start:start + size]
                        step = lr / (1 + t / 100) if "schedule" in name else lr
                        w = w - step * 2 / len(X[idx]) * X[idx].T @ (X[idx] @ w - y[idx])
                        t += 1
                    loss = np.mean((X @ w - y) ** 2)
                    if not np.isfinite(loss) or loss > 1e6:
                        break
                    losses.append(loss)
                    path.append(w.copy())
                board.append({"learning rate": lr, "variant": name, "epochs run": len(losses),
                              "final MSE": losses[-1] if losses else np.inf,
                              "distance to OLS": np.linalg.norm(w - ols)})
                fig.add_trace(go.Scatter(y=losses, name=name, visible=li == 0, mode="lines",
                                         line=dict(width=3, color=COLORS[vi])), row=1, col=1)
                if size == n:
                    p = np.array(path)
                    fig.add_trace(go.Scatter(x=p[:, 0], y=p[:, 1], mode="lines+markers", name="batch path",
                                             visible=li == 0, marker=dict(size=5, color=COLORS[3]),
                                             line=dict(width=2, color=COLORS[3])), row=1, col=2)
        total = len(lrs) * per_lr
        buttons = [dict(label=f"η = {lr}", method="update",
                        args=[{"visible": [True] + [j // per_lr == i for j in range(total)]}])
                   for i, lr in enumerate(lrs)]
        fig.update_layout(template="plotly_white", height=500,
                          updatemenus=[dict(buttons=buttons, x=0, y=1.2, xanchor="left")],
                          xaxis_title="epoch", yaxis=dict(type="log", title="MSE"),
                          xaxis2_title="intercept", yaxis2_title="slope")
        fig.show()
        return check, pd.DataFrame(board).round(5)

## Data
`ID` and `ZIP.Code` are identifiers, so we drop them. A few rows have negative `Experience`, which is not possible, so those are set to 0. We keep 20% of the data aside as a test set and do all tuning on the other 80%.

In [ ]:
lab = LoanTreeLab("bankloan.csv")
lab.df.head()

In [ ]:
lab.df["Personal.Loan"].value_counts(normalize=True).round(3)

---
## Exploratory Data Analysis
Every chart below has a dropdown so you can switch what you're looking at. Green is customers who **took** the loan, gray is customers who didn't.

In [ ]:
df = lab.df.copy()
df["Loan"] = df["Personal.Loan"].map({0: "No loan", 1: "Loan"})
palette = {"No loan": "#9ca3af", "Loan": "#10b981"}

### 1. How many customers took the loan?

In [ ]:
counts = df["Loan"].value_counts()
fig = go.Figure(go.Pie(labels=counts.index, values=counts.values, hole=0.6,
                       marker=dict(colors=[palette[k] for k in counts.index]),
                       textinfo="label+percent", textfont_size=15))
fig.update_layout(template="plotly_white", height=380, showlegend=False,
                  title="Loan acceptance",
                  annotations=[dict(text=f"{len(df):,}<br>customers", showarrow=False, font_size=18)])
fig.show()

Only **9.6%** of customers accepted. A model that always says "no" would already be 90% accurate, which is why we judge the tree on F1 instead of accuracy.

### 2. Numeric features: loan vs no loan
**Use the dropdown** to pick a feature. Each histogram is normalised so both groups can be compared even though one is ten times bigger.

In [ ]:
numeric = ["Income", "CCAvg", "Mortgage", "Age", "Experience"]
fig = go.Figure()
for i, col in enumerate(numeric):
    for label in ["No loan", "Loan"]:
        fig.add_trace(go.Histogram(x=df.loc[df["Loan"] == label, col], name=label, visible=i == 0,
                                   histnorm="percent", nbinsx=40, opacity=0.75,
                                   marker_color=palette[label]))
buttons = [dict(label=col, method="update",
                args=[{"visible": [j // 2 == i for j in range(len(numeric) * 2)]},
                      {"xaxis.title.text": col}])
           for i, col in enumerate(numeric)]
fig.update_layout(template="plotly_white", height=450, barmode="overlay", bargap=0.05,
                  title="Distribution by loan status", xaxis_title="Income", yaxis_title="% of group",
                  updatemenus=[dict(buttons=buttons, x=0, y=1.18, xanchor="left")])
fig.show()

- **Income** is the clearest signal: the median loan taker earns **$142.5k**, versus **$59k** for everyone else.
- **CCAvg** tells the same story: $3.8k a month vs $1.4k.
- **Age** and **Experience** look almost identical for both groups, so they won't help much.

### 3. Acceptance rate by customer group
**Use the dropdown** to pick a category.

In [ ]:
categories = {"Education": {1: "Undergrad", 2: "Graduate", 3: "Advanced"},
              "Family": {1: "1", 2: "2", 3: "3", 4: "4"},
              "CD.Account": {0: "No", 1: "Yes"},
              "Securities.Account": {0: "No", 1: "Yes"},
              "Online": {0: "No", 1: "Yes"},
              "CreditCard": {0: "No", 1: "Yes"}}
fig = go.Figure()
for i, (col, names) in enumerate(categories.items()):
    rate = df.groupby(col)["Personal.Loan"].mean()
    fig.add_trace(go.Bar(x=[names[k] for k in rate.index], y=rate.values, visible=i == 0,
                         text=[f"{v:.1%}" for v in rate.values], textposition="outside",
                         marker_color="#10b981", name=col))
fig.add_hline(y=df["Personal.Loan"].mean(), line_dash="dash", line_color="gray",
              annotation_text="overall 9.6%")
buttons = [dict(label=col, method="update",
                args=[{"visible": [j == i for j in range(len(categories))]}, {"xaxis.title.text": col}])
           for i, col in enumerate(categories)]
fig.update_layout(template="plotly_white", height=450, showlegend=False,
                  title="Loan acceptance rate", xaxis_title="Education",
                  yaxis=dict(title="acceptance rate", tickformat=".0%", range=[0, 0.55]),
                  updatemenus=[dict(buttons=buttons, x=0, y=1.18, xanchor="left")])
fig.show()

- Customers with a **CD account** accept **46%** of the time, almost 5× the average.
- Graduate and advanced degree holders accept about **3×** more often than undergrads (13% vs 4%).
- Online banking, credit cards and securities accounts make almost no difference.

### 4. Income vs credit card spend
**Use the dropdown** to filter by education level.

In [ ]:
levels = {"All": None, "Undergrad": 1, "Graduate": 2, "Advanced": 3}
fig = go.Figure()
for i, (name, level) in enumerate(levels.items()):
    sub = df if level is None else df[df["Education"] == level]
    for label in ["No loan", "Loan"]:
        part = sub[sub["Loan"] == label]
        fig.add_trace(go.Scattergl(x=part["Income"], y=part["CCAvg"], mode="markers", name=label,
                                   visible=i == 0, marker=dict(color=palette[label], size=6,
                                   opacity=0.45 if label == "No loan" else 0.85)))
buttons = [dict(label=name, method="update",
                args=[{"visible": [j // 2 == i for j in range(len(levels) * 2)]}])
           for i, name in enumerate(levels)]
fig.update_layout(template="plotly_white", height=500, title="Income vs CCAvg",
                  xaxis_title="Income ($000)", yaxis_title="CCAvg ($000 / month)",
                  updatemenus=[dict(buttons=buttons, x=0, y=1.15, xanchor="left")])
fig.show()

Almost nobody earning under **$100k** takes the loan (1.1%), while above $100k it jumps to **36%**. Switch to *Undergrad* and the green points nearly disappear. This is exactly the kind of clean split a decision tree is good at finding.

### 5. Correlation with the target

In [ ]:
corr = df.drop(columns="Loan").corr()["Personal.Loan"].drop("Personal.Loan").sort_values()
fig = go.Figure(go.Bar(x=corr.values, y=corr.index, orientation="h",
                       marker_color=["#10b981" if v > 0.1 else "#9ca3af" for v in corr.values],
                       text=[f"{v:.2f}" for v in corr.values], textposition="outside"))
fig.update_layout(template="plotly_white", height=450, title="Correlation with Personal.Loan",
                  xaxis=dict(title="correlation", range=[-0.1, 0.6]))
fig.show()

Income (0.50), CCAvg (0.37) and CD.Account (0.32) stand out. Age and Experience are almost 0, and they are 0.99 correlated with each other, so they carry the same information.

**EDA takeaway:** income, spending, education and a CD account drive loan acceptance. We expect the tree to split on these first.

---
## Member 1: Cross-validation
**Use the dropdown** to switch between split criterion (gini / entropy) and metric (F1, accuracy, ROC AUC). Hover to compare train and CV at each depth.

**Prediction before running:** training score will keep going up with depth, but the CV score will peak somewhere around depth 4–6 and then drop.

In [ ]:
cv_table, grid = lab.cross_validation(depths=range(1, 16), k=5)
cv_table

In [ ]:
print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 4))

**What we found**
- Accuracy is misleading here. A depth-1 tree gets 90.4% accuracy but an F1 of 0 because it just predicts "no loan" for everyone.
- CV F1 peaks at **depth 4 with entropy (0.914)**. After that the tree keeps fitting the training data better, but the CV score slowly drops. That gap is overfitting.
- ROC AUC drops even faster: from 0.994 at depth 3 to about 0.93 at depth 15.
- GridSearchCV picked the same depth we saw in the chart.

**Talking points**
1. `StratifiedKFold(n_splits=5, shuffle=True)` keeps the 9.6% "yes" rate the same in every fold.
2. `cross_validate(..., return_train_score=True)` gives both train and CV scores so we can see the gap.
3. `GridSearchCV(..., scoring="f1")` chooses depth and criterion using F1 instead of accuracy.

---
## Member 2: Loss minimization
Left chart: a model that predicts one constant value w₀ for everyone. We plot MSE and MAE for every w₀, with and without one extreme outlier (CCAvg = 40) added to a sample of 60 customers.
Right chart: a hand-coded decision stump. We try every possible Income threshold and keep the one with the lowest total SSE.

**Use the dropdown** to filter by education level. Click legend items to hide or show curves.

**Prediction before running:** the outlier will move the MSE minimum a lot and the MAE minimum barely at all.

In [ ]:
loss_summary = lab.loss_minimization(feature="Income", target="CCAvg", n=60, outlier=40)
loss_summary

**What we found**
- MSE is minimised at the **mean**, MAE at the **median**. With all customers, one outlier moved the MSE minimum from 1.78 to 2.40, but the MAE minimum only from 1.60 to 1.70.
- MSE is a smooth parabola because the squared error is smooth. MAE has kinks at every data point because |y − w₀| changes slope there.
- Our hand-coded stump splits at **Income = 116.5**, exactly the same threshold as `DecisionTreeRegressor(max_depth=1)` in every education group. A decision tree is just this loop repeated.

**Talking points**
1. `((ys[:, None] - w0) ** 2).mean(axis=0)` computes MSE for every w₀ at once.
2. `thresholds = (xs[:-1] + xs[1:]) / 2` gives the midpoints the tree also tests.
3. `.tree_.threshold[0]` reads the split scikit-learn chose so we can compare.

---
## Member 3: Ordinary Least Squares
We build X = [1, z, z²] where z is standardized income, and solve β̂ = (XᵀX)⁻¹Xᵀy with NumPy. Then we compare it with scikit-learn.

**Use the slider** to change the polynomial degree. The right chart shows how badly conditioned XᵀX becomes with raw vs standardized income (log scale).

**Prediction before running:** the normal equation will match scikit-learn exactly, but raw income with degree 8 will break.

In [ ]:
coefs, cond_table, leaf_check = lab.normal_equation(feature="Income", target="CCAvg", max_degree=8)
coefs

In [ ]:
cond_table

In [ ]:
leaf_check

**What we found**
- The normal equation and `LinearRegression` give the same coefficients to 6 decimals. scikit-learn is solving the same equation.
- With raw income, cond(XᵀX) goes from 2.7×10⁴ at degree 1 to about **9×10³³ at degree 8**. With standardized income it stays around 6×10⁷.
- From degree 6 onward `inv()` and `lstsq()` disagree, and lstsq's degree-8 training MSE (1.98) is worse than degree 5 (1.76). A higher degree should never fit training data worse, so the numbers are no longer trustworthy. Standardizing fixes this.
- Every leaf value of a depth-2 regression tree equals the intercept-only OLS solution for the points in that leaf, which is just their mean.

**Talking points**
1. `np.column_stack([np.ones_like(z), z, z ** 2])` builds the design matrix.
2. `np.linalg.inv(X.T @ X) @ X.T @ y` is the normal equation in one line.
3. `np.linalg.cond(Xr.T @ Xr)` shows when the matrix is too unstable to invert.

---
## Member 4: Gradient descent
We fit the same line (CCAvg vs standardized Income) with gradient descent written from scratch, then compare batch, mini-batch (size 8), SGD, and SGD with a decaying learning rate η_t = η₀ / (1 + t/100).

**Use the dropdown** to change the learning rate. The left chart shows the loss per epoch, the right chart shows the path batch GD takes across the loss surface.

**Prediction before running:** a small η will be slow, a large η will make SGD blow up.

In [ ]:
gd_check, leaderboard = lab.gradient_descent(feature="Income", target="CCAvg",
                                             lrs=(0.001, 0.01, 0.1, 0.5), epochs=60, batch=8)
gd_check

In [ ]:
leaderboard.sort_values("final MSE")

**What we found**
- Batch GD (η = 0.1, stop when gradient norm < 1e-6) matches OLS to 4 decimals: intercept 1.9379, slope 1.1288.
- With η = 0.001, batch GD is still far from the answer after 60 epochs (MSE 5.74 vs 1.78).
- With η = 0.5, plain SGD diverges straight away, but SGD with the decaying schedule still lands within 0.01 of OLS.
- Mini-batch and SGD bounce around the minimum with a fixed η. The schedule lets them settle.

**Talking points**
1. `grad = 2 / n * X.T @ (X @ w - y)` is the gradient of MSE.
2. `w -= 0.1 * grad` is the whole update step.
3. `order[start:start + size]` is the only change between batch, mini-batch and SGD.

---
## Final model
We use the tree GridSearchCV picked (entropy, depth 4) and score it once on the test set we kept aside at the start.
**Use the dropdown** on the confusion matrix to switch between counts and percentages.

In [ ]:
model = lab.best_tree
pred = model.predict(lab.X_test)
pd.Series({
    "accuracy": accuracy_score(lab.y_test, pred),
    "f1": f1_score(lab.y_test, pred),
    "roc_auc": roc_auc_score(lab.y_test, model.predict_proba(lab.X_test)[:, 1]),
}).round(4)

In [ ]:
cm = confusion_matrix(lab.y_test, pred)
labels = ["No loan", "Loan"]
fig = go.Figure([
    go.Heatmap(z=cm, x=labels, y=labels, text=cm, texttemplate="%{text}", colorscale="Blues", showscale=False),
    go.Heatmap(z=cm / cm.sum(axis=1, keepdims=True), x=labels, y=labels, texttemplate="%{z:.1%}",
               colorscale="Blues", showscale=False, visible=False),
])
fig.update_layout(template="plotly_white", height=420, width=520, title="Confusion matrix (test set)",
                  xaxis_title="Predicted", yaxis_title="Actual", yaxis_autorange="reversed",
                  updatemenus=[dict(x=0, y=1.2, xanchor="left", buttons=[
                      dict(label="Counts", method="update", args=[{"visible": [True, False]}]),
                      dict(label="Percent", method="update", args=[{"visible": [False, True]}])])])
fig.show()

In [ ]:
importance = pd.Series(model.feature_importances_, lab.X_train.columns).sort_values()
fig = go.Figure(go.Bar(x=importance.values, y=importance.index, orientation="h", marker_color="#2563eb"))
fig.update_layout(template="plotly_white", height=420, title="Feature importance",
                  xaxis_title="importance",
                  updatemenus=[dict(x=0, y=1.15, xanchor="left", buttons=[
                      dict(label="All features", method="relayout", args=[{"yaxis.range": [-0.5, len(importance) - 0.5]}]),
                      dict(label="Top 5", method="relayout", args=[{"yaxis.range": [len(importance) - 5.5, len(importance) - 0.5]}])])])
fig.show()

**Result:** on 1,000 unseen customers the tree scored **99.2% accuracy, F1 0.958, ROC AUC 0.993**. It missed 5 of 96 loan takers and wrongly flagged 3 customers.
Income does most of the work (59% of importance), then Education, Family size and CCAvg. Seven features are not used at all.

**Our choice:** a depth-4 entropy tree, because it had the highest 5-fold CV F1 and deeper trees only overfit.

---
## Peer review Q&A

| # | Question from reviewer | Our answer |
|---|---|---|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |

**Reviewer's attempt to beat our test score** (change one hyperparameter):

| Change made | Test F1 |
|---|---|
|  |  |

**What we would change:**